In [25]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/ask-plfs")
for sub in ["src", "app", "data/raw", "data/processed", "index", "eval", "notebooks"]:
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "src" / "__init__.py").touch()   # makes src a Python package

os.chdir(PROJECT_DIR)                 # work from the project root
sys.path.insert(0, str(PROJECT_DIR))  # lets us write: from src import config
print("Working in:", os.getcwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working in: /content/drive/MyDrive/ask-plfs


In [26]:
import importlib, sys

def reload_src():
    """Reload all src modules, in dependency order, after editing a file."""
    for name in ["src.config", "src.utils", "src.ingest", "src.chunking",
                 "src.indexing", "src.retrieval", "src.generation",
                 "src.pipeline", "src.evaluation"]:
        if name in sys.modules:
            importlib.reload(sys.modules[name])
    print("src modules reloaded")

In [3]:
   !pip -q install pymupdf pymupdf4llm sentence-transformers faiss-cpu rank-bm25 google-genai pandas streamlit

In [27]:
import os
from google.colab import userdata
os.environ["GeminiAPIkey"] = userdata.get("GeminiAPIkey")


In [63]:
%%writefile src/config.py
from pathlib import Path

# ---------- Paths ----------
ROOT = Path(__file__).resolve().parent.parent
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
INDEX_DIR = ROOT / "index"
PAGES_PATH = DATA_PROCESSED / "pages.jsonl"
CHUNKS_PATH = DATA_PROCESSED / "chunks.jsonl"
FAISS_PATH = INDEX_DIR / "chunks.faiss"
EVAL_PATH = ROOT / "eval" / "eval_questions.csv"

# ---------- Corpus ----------
REPORTS = {
    "plfs_annual_report_2025.pdf": {
        "label": "PLFS Annual Report 2025",
        "url": "https://raw.githubusercontent.com/mani-aditya/file_dump/main/plfs2025report.pdf",
    },
}

# ---------- Parsing and chunking ----------
CHUNK_SIZE_TOKENS = 256      # the embedding model reads at most 512; this leaves headroom
CHUNK_OVERLAP_TOKENS = 40
MIN_PAGE_CHARS = 200          # skip near-empty pages
MIN_ENGLISH_RATIO = 0.8       # skip Hindi / garbled pages
USE_MARKDOWN = True

# ---------- Embeddings ----------
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

# ---------- Retrieval and generation ----------
TOP_K = 8
LLM_MODEL = "gemini-3.1-flash-lite"
FALLBACK_MODEL = "gemini-3.8-flash"
MAX_ANSWER_TOKENS = 800

Overwriting src/config.py


In [30]:
%%writefile src/utils.py
import json
from pathlib import Path


def save_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

Overwriting src/utils.py


In [31]:
%%writefile src/ingest.py
import re
import urllib.request

import pymupdf

from src import config
from src.utils import save_jsonl
# matches "Annual Report, PLFS, 2025 xx" and "Annual Report, PLFS, 2025 10" (printed page number)
PRINTED_RE = re.compile(r"Annual Report, PLFS, 2025\s+([ivxlc]+|\d+)\b")
# first line that looks like "Table 4: ..." is used as the page title
TITLE_RE = re.compile(r"^[#*\s]*((?:Table|Statement|Annexure)\b[^\n]{0,150})", re.M)
HEADING_RE = re.compile(r"^#{1,6}\s*(.+?)\s*$", re.M)   # fallback title: first markdown heading

def clean_text(text):
    text = text.replace("\x00", " ").replace("<br>", " ")
    text = re.sub(r"</?mark>", "", text)                       # parser highlight tags
    text = text.replace("**", "")                              # bold markers
    text = re.sub(r"^\|[-:| ]+\|\s*$", "", text, flags=re.M)   # drop |---|---| rows (must come BEFORE the dash line)
    text = re.sub(r"-{3,}", " ", text)                         # long dash runs
    text = re.sub(r"(\|[ \t]*){2,}", "| ", text)               # empty cells; [ \t] not \s, so rows stay on separate lines
    text = re.sub(r"-\n(?=[a-z])", "", text)                   # re-join hyphenated words
    text = re.sub(r"^_Summar[^\n]*$", "", text, flags=re.M)    # garbled running header
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def get_page_texts(pdf_path):
    """One text string per PDF page."""
    if config.USE_MARKDOWN:
        import pymupdf4llm
        parts = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
        return [part["text"] for part in parts]
    doc = pymupdf.open(pdf_path)
    return [page.get_text("text", sort=True) for page in doc]

def extract_pages(pdf_path, report_label):
    pages, skipped = [], []
    for page_index, raw in enumerate(get_page_texts(pdf_path)):
        text = clean_text(raw)
        match = PRINTED_RE.search(text)
        printed_page = match.group(1) if match else None
        text = PRINTED_RE.sub("", text).strip()          # remove the repeated header/footer
        title = TITLE_RE.search(text) or HEADING_RE.search(text)
        if len(text) < config.MIN_PAGE_CHARS:
            skipped.append((page_index + 1, "too short")); continue
        if english_ratio(text) < config.MIN_ENGLISH_RATIO:
            skipped.append((page_index + 1, "not English")); continue
        pages.append({
            "report": report_label,
            "source_file": pdf_path.name,
            "page": page_index + 1,                      # PDF page
            "printed_page": printed_page,                # page number printed on the page
            "title": title.group(1).strip(" *#") if title else None,
            "text": text,
        })
    print(f"{pdf_path.name}: kept {len(pages)}, skipped {len(skipped)}")
    print("First skipped pages:", skipped[:20])
    return pages

def download_pdf(url, dest):
    """Download a PDF, pretending to be a normal browser."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        dest.write_bytes(response.read())





def english_ratio(text):
    """Share of letters that are plain ASCII (Hindi text scores near 0)."""
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    return sum(c.isascii() for c in letters) / len(letters)





def build_pages():
    """Download (if missing) and parse every report listed in config.REPORTS."""
    all_pages = []
    for file_name, meta in config.REPORTS.items():
        pdf_path = config.DATA_RAW / file_name
        if not pdf_path.exists():
            print(f"Downloading {file_name} ...")
            download_pdf(meta["url"], pdf_path)
        pages = extract_pages(pdf_path, meta["label"])

        all_pages.extend(pages)
    save_jsonl(all_pages, config.PAGES_PATH)
    return all_pages

Overwriting src/ingest.py


In [32]:
import sys
import importlib

# Force Python to reload the updated config and ingest modules
if "src.config" in sys.modules:
    importlib.reload(sys.modules["src.config"])
if "src.ingest" in sys.modules:
    importlib.reload(sys.modules["src.ingest"])

from src import ingest
pages = ingest.build_pages()
print("Total pages loaded:", len(pages))
by_page = {p["page"]: p for p in pages}

for n in [22, 25, 40]:
    p = by_page.get(n)
    if p is None:
        print(f"--- PDF page {n}: not in pages (skipped) ---")
        continue
    print(f"--- PDF page {n} | printed: {p['printed_page']} | title: {p['title']} ---")
    print(p["text"][:800])
    print()


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=2/3.
OCR on page.number=3/4.
OCR on page.number=21/22.
OCR on page.number=24/25.
OCR on page.number=583/584.
OCR on page.number=589/590.
OCR on page.number=590/591.
OCR on page.number=593/594.
OCR on page.number=598/599.
OCR on page.number=599/600.
OCR on page.number=605/606.
OCR on page.number=652/653.
plfs_annual_report_2025.pdf: kept 624, skipped 29
First skipped pages: [(2, 'not English'), (6, 'not English'), (7, 'not English'), (8, 'not English'), (9, 'not English'), (10, 'not English'), (11, 'not English'), (12, 'not English'), (13, 'not English'), (14, 'not English'), (15, 'not English'), (16, 'not English'), (29, 'too short'), (30, 'too short'), (73, 'too short'), (74, 'too short'), (525, 'too short'), (526, 'too short'), (546, 'too short'), (547, 'too short')]
Total pages loaded: 624
--- PDF page 22 | printed: None | title: Table 1: Labour Force Participation Rate (

In [33]:
%%writefile src/chunking.py
from pathlib import Path
from transformers import AutoTokenizer
from src import config
from src.utils import load_jsonl, save_jsonl

_tokenizer = None

def get_tokenizer():
    global _tokenizer
    if _tokenizer is None:
        _tokenizer = AutoTokenizer.from_pretrained(config.EMBEDDING_MODEL)
    return _tokenizer

def split_into_chunks(text, chunk_size, overlap):
    """Pack whole lines (table rows / paragraphs) into chunks of <= chunk_size tokens.
    A single line longer than chunk_size (a long paragraph) is cut by tokens instead."""
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if not lines:
        return []
    enc = get_tokenizer()(lines, add_special_tokens=False,
                          return_offsets_mapping=True, verbose=False)
    pieces = []                                   # (text, n_tokens)
    for line, ids, offs in zip(lines, enc["input_ids"], enc["offset_mapping"]):
        n = len(ids)
        if n <= chunk_size:
            pieces.append((line, n))
            continue
        for s in range(0, n, chunk_size - overlap):
            e = min(s + chunk_size, n)
            pieces.append((line[offs[s][0]:offs[e - 1][1]], e - s))
            if e == n:
                break

    chunks, current, size = [], [], 0
    for line, n in pieces:
        if current and size + n > chunk_size:
            chunks.append("\n".join(t for t, _ in current))
            last = current[-1]
            current = [last] if last[1] <= overlap else []   # repeat last short line for continuity
            size = sum(k for _, k in current)
        current.append((line, n))
        size += n
    if current:
        chunks.append("\n".join(t for t, _ in current))
    return chunks

def build_chunks():
    pages = load_jsonl(config.PAGES_PATH)
    chunks = []
    for page in pages:
        pieces = split_into_chunks(
            page["text"], config.CHUNK_SIZE_TOKENS, config.CHUNK_OVERLAP_TOKENS
        )
        rows = [l.strip() for l in page["text"].split("\n") if l.strip().startswith("|")]
        table_head = "\n".join(rows[:2])          # the table's column-header rows
        header = f"{page['report']} | PDF page {page['page']}"
        if page.get("title"):
            header += f" | {page['title']}"
        stem = Path(page["source_file"]).stem
        for i, piece in enumerate(pieces):
            if i > 0 and table_head and piece.startswith("|"):
                piece = table_head + "\n" + piece   # later table chunks keep their column headers
            chunks.append({
                "chunk_id": f"{stem}_p{page['page']:03d}_c{i:02d}",
                "report": page["report"],
                "source_file": page["source_file"],
                "page": page["page"],
                "printed_page": page.get("printed_page"),
                "text": header + "\n" + piece,
            })
    save_jsonl(chunks, config.CHUNKS_PATH)
    print(f"{len(pages)} pages -> {len(chunks)} chunks")
    return chunks

Overwriting src/chunking.py


In [34]:
reload_src()
from src import chunking
chunks = chunking.build_chunks()
print(chunks[50])

src modules reloaded
624 pages -> 4437 chunks
{'chunk_id': 'plfs_annual_report_2025_p034_c01', 'report': 'PLFS Annual Report 2025', 'source_file': 'plfs_annual_report_2025.pdf', 'page': 34, 'printed_page': None, 'text': 'PLFS Annual Report 2025 | PDF page 34\n|Table 1(b): Modi December 2025 w|fications in the coverage of the items of info ith respect to Schedule 10.4 canvassed during |rmation of Schedule 10.4 during July- January-June 2025|\n|Reference to Schedule|PLFS Schedule canvassed during January- June 2025|PLFS Schedule canvassed during July – December 2025|\n|Reference to Schedule|PLFS Schedule canvassed during January- June 2025|PLFS Schedule canvassed during July – December 2025|\n|_Columns 15 of_ _Block_ _4_ _of_ _Schedule_ _10.4_ _(FV & RV)_|Person-level data on Column-15 (status of current attendance in educational institution) was collected for persons aged 0 to 29 years.|Person-level data on Column-15 (status of current attendance in educational institution) was collecte

In [35]:
from src import config
from sentence_transformers import SentenceTransformer
tok = SentenceTransformer(config.EMBEDDING_MODEL).tokenizer
lengths = [len(tok.encode(c["text"])) for c in chunks]
print("longest:", max(lengths), "| over 512:", sum(l > 512 for l in lengths), "of", len(lengths))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (607 > 512). Running this sequence through the model will result in indexing errors


longest: 695 | over 512: 11 of 4437


In [36]:
%%writefile src/indexing.py
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

from src import config
from src.utils import load_jsonl


def load_embedder():
    return SentenceTransformer(config.EMBEDDING_MODEL)


def embed_texts(model, texts, batch_size=64):
    vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return np.asarray(vectors, dtype="float32")


def build_faiss_index(vectors):
    index = faiss.IndexFlatIP(vectors.shape[1])   # inner product on normalised vectors = cosine
    index.add(vectors)
    return index


def save_index(index, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(path))


def load_index(path):
    return faiss.read_index(str(path))


def build_and_save_index():
    chunks = load_jsonl(config.CHUNKS_PATH)
    model = load_embedder()
    vectors = embed_texts(model, [c["text"] for c in chunks])
    index = build_faiss_index(vectors)
    save_index(index, config.FAISS_PATH)
    print(f"Indexed {index.ntotal} chunks, dimension {vectors.shape[1]}")
    return index

Overwriting src/indexing.py


In [37]:
from src import config, indexing
REBUILD = False      # set to False once your chunks are final
if REBUILD or not config.FAISS_PATH.exists():
    index = indexing.build_and_save_index()
else:
    index = indexing.load_index(config.FAISS_PATH)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Indexed 4437 chunks, dimension 384


In [38]:
%%writefile src/retrieval.py
import re

import numpy as np
from rank_bm25 import BM25Okapi

from src import config, indexing
from src.utils import load_jsonl


def tokenize(text):
    """Lowercase words and numbers; keeps decimals like 41.7 as one token."""
    return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", text.lower())


class Retriever:
    def __init__(self):
        self.chunks = load_jsonl(config.CHUNKS_PATH)
        self.index = indexing.load_index(config.FAISS_PATH)
        self.embedder = indexing.load_embedder()
        assert self.index.ntotal == len(self.chunks), "Index and chunks are out of sync - rebuild the index"
        self.bm25 = BM25Okapi([tokenize(c["text"]) for c in self.chunks])

    # ---- the three search modes: each returns [(chunk_row, score), ...] ----
    def dense_search(self, query, k):
        vector = self.embedder.encode(
            [config.QUERY_PREFIX + query], normalize_embeddings=True
        ).astype("float32")
        scores, ids = self.index.search(vector, k)
        return [(int(i), float(s)) for i, s in zip(ids[0], scores[0]) if i != -1]

    def bm25_search(self, query, k):
        scores = self.bm25.get_scores(tokenize(query))
        top_ids = np.argsort(scores)[::-1][:k]
        return [(int(i), float(scores[i])) for i in top_ids]

    def hybrid_search(self, query, k, candidates=20, rrf_k=60):
        dense = self.dense_search(query, candidates)
        sparse = self.bm25_search(query, candidates)
        fused = {}
        for results in (dense, sparse):
            for rank, (idx, _) in enumerate(results):
                fused[idx] = fused.get(idx, 0.0) + 1.0 / (rrf_k + rank + 1)
        ranked = sorted(fused.items(), key=lambda item: item[1], reverse=True)
        return ranked[:k]

    def _limit_per_page(self, hits, k, max_per_page=2):
       kept, per_page = [], {}
       for idx, score in hits:
           page = self.chunks[idx]["page"]
           if per_page.get(page, 0) >= max_per_page:
               continue
           per_page[page] = per_page.get(page, 0) + 1
           kept.append((idx, score))
           if len(kept) == k:
               break
       return kept

    # ---- the public method everything else calls ----
    def search(self, query, k=config.TOP_K, mode="hybrid", max_per_page=2):
       wide = k * 3          # fetch extra, then trim so one page can't fill every slot
       if mode == "dense":
           hits = self.dense_search(query, wide)
       elif mode == "bm25":
           hits = self.bm25_search(query, wide)
       elif mode == "hybrid":
           hits = self.hybrid_search(query, wide)
       else:
           raise ValueError("mode must be 'dense', 'bm25' or 'hybrid'")
       hits = self._limit_per_page(hits, k, max_per_page)
       return [{**self.chunks[idx], "score": score} for idx, score in hits]

Overwriting src/retrieval.py


In [39]:
from src.retrieval import Retriever
retriever = Retriever()

query = "What was the female labour force participation rate in 2025?"
for mode in ["dense", "bm25", "hybrid"]:
    print(f"\n=== {mode} ===")
    for hit in retriever.search(query, k=3, mode=mode):
        print(f"page {hit['page']} | score {hit['score']:.3f} | {hit['text'][:150]}...")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


=== dense ===
page 41 | score 0.783 | PLFS Annual Report 2025 | PDF page 41 | Statement 2: Labour force participation rates (in per cent) in usual status (ps+ss) estimated from PLFS 2022, ...
page 25 | score 0.772 | PLFS Annual Report 2025 | PDF page 25 | Table 4: Labour Force Participation Rate (in per cent) in current weekly status estimated from PLFS 2022, PLFS...
page 41 | score 0.763 | PLFS Annual Report 2025 | PDF page 41 | Statement 2: Labour force participation rates (in per cent) in usual status (ps+ss) estimated from PLFS 2022, ...

=== bm25 ===
page 18 | score 23.418 | PLFS Annual Report 2025 | PDF page 18 | A. Labour Force in usual status (ps+ss)
_Highlights_
Some of the key results at the all-India level for the pe...
page 584 | score 22.886 | PLFS Annual Report 2025 | PDF page 584 | Appendix - D
_Sample Design and Estimation Procedure_
<!-- Start of picture text -->
—sssssssssssssssssssssO0...
page 40 | score 21.683 | PLFS Annual Report 2025 | PDF page 40 | 2.1.1 Labour 

In [65]:
%%writefile src/generation.py
import os
import time

from google import genai
from google.genai import types

from src import config

SYSTEM_PROMPT = """You are a careful research assistant answering questions about India's Periodic Labour Force Survey (PLFS) Annual Report 2025.

Rules:
1. Use ONLY the numbered context passages provided. Do not use outside knowledge.
2. Cite every factual claim with the passage number in square brackets, e.g. [1] or [2].
3. If the passages do not contain the answer, reply exactly: "I could not find this in the provided PLFS reports."
4. Quote numbers exactly as written, with their units and reference period (for example 2025, usual status ps+ss, age group).
5. Many passages are table fragments. If you cannot tell which row or column a number belongs to, do not guess: say the table layout is unclear and cite the passage.
6. If the question can be read several ways (usual status vs current weekly status, rural vs urban, male vs female), give each relevant figure briefly, clearly labelled.
7. Keep the answer short and precise."""

RETRY_CODES = {429, 500, 502, 503, 504}   # rate limit / server overloaded: worth retrying


def format_context(passages):
    blocks = []
    for number, p in enumerate(passages, start=1):
        where = f"PDF page {p['page']}"
        if p.get("printed_page"):
            where += f", printed page {p['printed_page']}"
        blocks.append(f"[{number}] ({p['report']}, {where})\n{p['text']}")
    return "\n\n".join(blocks)


_client = None

def get_client(api_key=None):
    """Create the Gemini client once and reuse it."""
    global _client
    key = api_key or os.environ.get("GeminiAPIkey") or os.environ.get("GEMINI_API_KEY")
    if not key:
        raise ValueError("No API key found. Set GeminiAPIkey or GEMINI_API_KEY.")
    if _client is None:
        _client = genai.Client(api_key=key, http_options=types.HttpOptions(timeout=30_000))  # 30 s, in milliseconds
    return _client


def build_user_prompt(question, passages):
    return f"Context passages:\n\n{format_context(passages)}\n\nQuestion: {question}"


def _call_model(client, model, prompt):
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=config.MAX_ANSWER_TOKENS,
            temperature=0.0,
            thinking_config=types.ThinkingConfig(thinking_budget=0),
        ),
    )
    return response.text or "The model returned no text. Please try again."


def generate_answer(question, passages, api_key=None, retries=2):
    client = get_client(api_key)
    prompt = build_user_prompt(question, passages)
    models = [m for m in (config.LLM_MODEL, config.FALLBACK_MODEL) if m]
    for model in models:
        for attempt in range(retries):
            try:
                return _call_model(client, model, prompt)
            except Exception as e:
                code = getattr(e, "code", None)
                timed_out = "timeout" in type(e).__name__.lower() or "timed out" in str(e).lower()
                if code not in RETRY_CODES and not timed_out:
                    raise                              # a real bug: show it
                wait = 5 * 2 ** attempt
                print(f"{model}: {'timeout' if timed_out else code}; retrying in {wait}s")
                time.sleep(wait)
        print(f"{model} still busy, trying the next model")
    return "The Gemini service is busy right now. Please try again in a minute."

Overwriting src/generation.py


In [66]:
%%writefile src/pipeline.py
from src import config, generation


def answer_question(retriever, question, k=config.TOP_K, mode="hybrid",
                    use_llm=True, api_key=None):
    passages = retriever.search(question, k=k, mode=mode)
    answer = generation.generate_answer(question, passages, api_key=api_key) if use_llm else None
    sources = [f"[{n}] {p['report']}, PDF page {p['page']}" for n, p in enumerate(passages, 1)]
    return {"question": question, "answer": answer, "passages": passages, "sources": sources}


def show(result):
    print("Q:", result["question"], "\n")
    print(result["answer"], "\n")
    print("Sources:")
    for s in result["sources"]:
        print("  ", s)

Overwriting src/pipeline.py


In [67]:
reload_src()
from src.pipeline import answer_question, show
show(answer_question(retriever, "What was the female labour force participation rate in 2025?"))

src modules reloaded
Q: What was the female labour force participation rate in 2025? 

Based on the PLFS Annual Report 2025 (referring to the period January 2025 – December 2025), the female Labour Force Participation Rate (LFPR) is as follows:

**Usual Status (ps+ss):**
*   **All ages:** 30.7% [3]
*   **15 years and above:** 40.0% [3]
*   **15-29 years:** 27.5% [2]

**Current Weekly Status (CWS):**
*   **All ages:** 26.3% [4]
*   **15 years and above:** 34.3% [4] 

Sources:
   [1] PLFS Annual Report 2025, PDF page 40
   [2] PLFS Annual Report 2025, PDF page 41
   [3] PLFS Annual Report 2025, PDF page 22
   [4] PLFS Annual Report 2025, PDF page 25
   [5] PLFS Annual Report 2025, PDF page 41
   [6] PLFS Annual Report 2025, PDF page 18
   [7] PLFS Annual Report 2025, PDF page 25
   [8] PLFS Annual Report 2025, PDF page 584


In [68]:
from src import config
from src.utils import load_jsonl
PAGES = load_jsonl(config.PAGES_PATH)

def find_pages(phrase, show=20):
    hits = [p["page"] for p in PAGES if phrase.lower() in p["text"].lower()]
    print(f"{len(hits)} pages contain {phrase!r}: {hits[:show]}")

find_pages("Table 1:")

5 pages contain 'Table 1:': [22, 595, 596, 597, 598]


In [69]:
import pandas as pd
from src import config

rows = [   # (question, PDF page(s) where the answer is, e.g. "40; 41")
    ("Who was considered literate for the purpose of the survey?", "549"),
    ("How is the term 'economic activity' defined in the PLFS", "551"),
]
pd.DataFrame(rows, columns=["question", "expected_pages"]).to_csv(config.EVAL_PATH, index=False)

In [70]:
%%writefile src/evaluation.py
import re
import pandas as pd


def parse_pages(value):
    """'40; 41' -> {40, 41}"""
    return {int(x) for x in re.findall(r"\d+", str(value))}


def evaluate_retrieval(retriever, eval_df, k=5, mode="hybrid"):
    eval_df = eval_df.dropna(subset=["question", "expected_pages"])
    if eval_df.empty:
        raise ValueError("No labelled questions - fill expected_pages and re-run the CSV cell")
    rows = []
    for _, row in eval_df.iterrows():
        wanted = parse_pages(row["expected_pages"])
        results = retriever.search(row["question"], k=k, mode=mode)
        rank = next((r for r, p in enumerate(results, start=1) if p["page"] in wanted), None)
        rows.append({"question": row["question"], "hit": rank is not None, "rank": rank})
    df = pd.DataFrame(rows)
    df["rank"] = pd.to_numeric(df["rank"])
    summary = {
        "mode": mode, "k": k, "n": len(df),
        "hit_rate": round(float(df["hit"].mean()), 3),
        "mrr": round(float((1 / df["rank"]).fillna(0).mean()), 3),
    }
    return df, summary

Overwriting src/evaluation.py


In [71]:
reload_src()
import pandas as pd
from src import config
from src.evaluation import evaluate_retrieval

eval_df = pd.read_csv(config.EVAL_PATH, dtype=str)
for mode in ["dense", "bm25", "hybrid"]:
    details, summary = evaluate_retrieval(retriever, eval_df, k=5, mode=mode)
    print(summary)
details   # shows which questions failed

src modules reloaded
{'mode': 'dense', 'k': 5, 'n': 2, 'hit_rate': 1.0, 'mrr': 1.0}
{'mode': 'bm25', 'k': 5, 'n': 2, 'hit_rate': 1.0, 'mrr': 1.0}
{'mode': 'hybrid', 'k': 5, 'n': 2, 'hit_rate': 1.0, 'mrr': 1.0}


,question,hit,rank
0,Who was considered literate for the purpose of...,True,1
1,How is the term 'economic activity' defined in...,True,1


In [72]:
for q in ["What was India's GDP growth rate in 2025?",
          "Who won the 2024 general election?",
          "What is the capital of France?"]:
    print(q, "->", answer_question(retriever, q)["answer"], "\n")

What was India's GDP growth rate in 2025? -> I could not find this in the provided PLFS reports. 

Who won the 2024 general election? -> I could not find this in the provided PLFS reports. 

What is the capital of France? -> I could not find this in the provided PLFS reports. 



In [74]:
show(answer_question(retriever, "An FSU may be a casualty for a particular second stage stratum although survey code is not what?"))

Q: An FSU may be a casualty for a particular second stage stratum although survey code is not what? 

An FSU may be a casualty for a particular second stage stratum although the survey code is not 7 [1]. 

Sources:
   [1] PLFS Annual Report 2025, PDF page 606
   [2] PLFS Annual Report 2025, PDF page 604
   [3] PLFS Annual Report 2025, PDF page 605
   [4] PLFS Annual Report 2025, PDF page 605
   [5] PLFS Annual Report 2025, PDF page 601
   [6] PLFS Annual Report 2025, PDF page 601
   [7] PLFS Annual Report 2025, PDF page 602
   [8] PLFS Annual Report 2025, PDF page 590
